# Qdrant Chunk ID + Section Audit (Per Paper)

This notebook helps you:
- retrieve all chunks for a paper from Qdrant
- print and save chunk IDs
- fetch exact chunk content by IDs
- list all `section_title` values
- print chunks grouped by section title

In [17]:
from pathlib import Path
import os

# Change this to the paper/document identifier you want to inspect.
PAPER_ID = "bd077a96-5a38-5281-993e-10cf869afcde"
PAPER_ID_FIELD = "document_id"
COLLECTION_NAME = os.getenv("QDRANT_COLLECTION", "research_papers")

# Optional limits for quick debugging.
SCROLL_BATCH_SIZE = 256
MAX_POINTS = None  # e.g., 100 for quick checks

PLAYGROUND_DIR = Path.cwd() if Path.cwd().name == "playground" else Path.cwd() / "playground"
PLAYGROUND_DIR.mkdir(parents=True, exist_ok=True)

CHUNK_IDS_JSON = PLAYGROUND_DIR / "chunk_ids.json"
CHUNK_IDS_CSV = PLAYGROUND_DIR / "chunk_ids.csv"
SECTIONS_JSON = PLAYGROUND_DIR / "sections_with_chunks.json"

print(f"Playground dir: {PLAYGROUND_DIR}")
print(f"Collection: {COLLECTION_NAME}")
print(f"Paper ID field: {PAPER_ID_FIELD}")
print(f"Paper ID: {PAPER_ID or '<set me>'}")

Playground dir: /home/aman/storage/Python/Projects/Research Paper Assistant/playground
Collection: research_papers
Paper ID field: document_id
Paper ID: bd077a96-5a38-5281-993e-10cf869afcde


In [18]:
# Install if needed.
%pip install -q qdrant-client pandas

import json
from collections import defaultdict

import pandas as pd
from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchValue

print("Imports ready.")

Note: you may need to restart the kernel to use updated packages.
Imports ready.


In [19]:
# Configure Qdrant connection via env vars.
# Cloud example: export QDRANT_URL='https://...:6333' and QDRANT_API_KEY='...'
# Local example: leave URL empty and set host/port below.
QDRANT_URL = os.getenv("QDRANT_URL", "")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY", "")
QDRANT_HOST = os.getenv("QDRANT_HOST", "localhost")
QDRANT_PORT = int(os.getenv("QDRANT_PORT", "6333"))

if QDRANT_URL:
    client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY or None)
else:
    client = QdrantClient(host=QDRANT_HOST, port=QDRANT_PORT)

collections = client.get_collections()
collection_names = [c.name for c in collections.collections]
print(f"Connected. Collections: {collection_names[:10]}")
if COLLECTION_NAME not in collection_names:
    print(f"Warning: '{COLLECTION_NAME}' not found in current Qdrant instance.")

Connected. Collections: ['research_papers', 'research_papers_main']


In [20]:
if not PAPER_ID:
    raise ValueError("Set PAPER_ID in the setup cell before running this cell.")

all_points = []
offset = None

while True:
    points, next_offset = client.scroll(
        collection_name=COLLECTION_NAME,
        scroll_filter=Filter(
            must=[
                FieldCondition(
                    key=PAPER_ID_FIELD,
                    match=MatchValue(value=PAPER_ID),
                )
            ]
        ),
        limit=SCROLL_BATCH_SIZE,
        with_payload=True,
        with_vectors=False,
        offset=offset,
    )

    if not points:
        break

    all_points.extend(points)

    if MAX_POINTS is not None and len(all_points) >= MAX_POINTS:
        all_points = all_points[:MAX_POINTS]
        break

    if next_offset is None:
        break

    offset = next_offset

chunk_records = []
for p in all_points:
    payload = p.payload if isinstance(p.payload, dict) else {}
    chunk_id = payload.get("chunk_id") or payload.get("_id")
    chunk_records.append(
        {
            "point_id": str(p.id),
            "chunk_id": str(chunk_id) if chunk_id is not None else "",
            "section_title": str(payload.get("section_title") or ""),
            "section_id": str(payload.get("section_id") or ""),
            "content": str(payload.get("content") or ""),
            "payload": payload,
        }
    )

print(f"Fetched points for paper: {PAPER_ID}")
print(f"Total points/chunks found: {len(chunk_records)}")

chunks_df = pd.DataFrame(chunk_records)
chunks_df.head(10)

Fetched points for paper: bd077a96-5a38-5281-993e-10cf869afcde
Total points/chunks found: 96


,point_id,chunk_id,section_title,section_id,content,payload
0,006558f7-0158-59e1-8a44-dea726b2cf43,12d0eec0-d010-4578-8595-95cc823bbe22,Reference,bd077a96-5a38-5281-993e-10cf869afcde_section_7,Diederik Kingma and Jimmy Ba. Adam: A method f...,{'chunk_id': '12d0eec0-d010-4578-8595-95cc823b...
1,0233c58f-c04c-5324-bf84-72532fdc1f24,a68ddde7-ac19-49ea-bae4-c82e5496244b,Reference,bd077a96-5a38-5281-993e-10cf869afcde_section_7,Smith. Recurrent neural network grammars. In P...,{'chunk_id': 'a68ddde7-ac19-49ea-bae4-c82e5496...
2,0278a119-a01b-539a-903f-65fa986c1bc1,2c4c7d66-0a9c-4c9d-b34e-5cb04da5df7a,Reference,bd077a96-5a38-5281-993e-10cf869afcde_section_7,Outrageously large neural networks: The sparse...,{'chunk_id': '2c4c7d66-0a9c-4c9d-b34e-5cb04da5...
3,033d9657-784e-521f-ae38-8ca7d78fac12,1f1078a1-7b2d-408c-bec1-fe9d039eb9f1,Reference,bd077a96-5a38-5281-993e-10cf869afcde_section_7,"ACL, August 2013. Jimmy Lei Ba, Jamie Ryan Kir...",{'chunk_id': '1f1078a1-7b2d-408c-bec1-fe9d039e...
4,0498359c-e94e-570a-8f23-692256a02687,c1af6cc1-824c-4122-9063-8c693ed63587,Reference,bd077a96-5a38-5281-993e-10cf869afcde_section_7,"ACL, June 2006. Ankur Parikh, Oscar Täckström,...",{'chunk_id': 'c1af6cc1-824c-4122-9063-8c693ed6...
5,07155d70-2e20-54aa-b25e-e6b10f079817,c2e45765-f41e-447f-95a6-0bd19be9244e,Reference,bd077a96-5a38-5281-993e-10cf869afcde_section_7,In Proceedings of the 2009 Conference on Empir...,{'chunk_id': 'c2e45765-f41e-447f-95a6-0bd19be9...
6,0a213754-813f-5499-b38c-1c27e77755b6,3afcf3a8-5919-484c-b39f-4b1bd2784cef,Reference,bd077a96-5a38-5281-993e-10cf869afcde_section_7,Smith. Recurrent neural network grammars. In P...,{'chunk_id': '3afcf3a8-5919-484c-b39f-4b1bd278...
7,15fa383b-bc84-5247-a86b-73d1380b2e80,ce13c9e1-6cbd-42a4-9c15-294a99f3bde1,Results,bd077a96-5a38-5281-993e-10cf869afcde_section_5,Table 3: Variations on the Transformer archite...,{'chunk_id': 'ce13c9e1-6cbd-42a4-9c15-294a99f3...
8,1937c5f0-a3a0-5230-9fa9-7fbd5af1f033,058aa52a-92f3-4686-bf1b-b4ab7a19167f,Why Self-Attention,bd077a96-5a38-5281-993e-10cf869afcde_section_3,Hence we also compare the maximum path length ...,{'chunk_id': '058aa52a-92f3-4686-bf1b-b4ab7a19...
9,1a540631-bff4-5989-94df-aae4f6e5a85a,6fc7f0ae-d7bd-4feb-a776-8d38af5b5ed0,Conclusion,bd077a96-5a38-5281-993e-10cf869afcde_section_6,We are excited about the future of attention-b...,{'chunk_id': '6fc7f0ae-d7bd-4feb-a776-8d38af5b...


In [21]:
chunk_ids = [r["chunk_id"] for r in chunk_records if r["chunk_id"]]

print(f"Chunk IDs count: {len(chunk_ids)}")
for i, cid in enumerate(chunk_ids, start=1):
    print(f"{i:04d}: {cid}")

Chunk IDs count: 96
0001: 12d0eec0-d010-4578-8595-95cc823bbe22
0002: a68ddde7-ac19-49ea-bae4-c82e5496244b
0003: 2c4c7d66-0a9c-4c9d-b34e-5cb04da5df7a
0004: 1f1078a1-7b2d-408c-bec1-fe9d039eb9f1
0005: c1af6cc1-824c-4122-9063-8c693ed63587
0006: c2e45765-f41e-447f-95a6-0bd19be9244e
0007: 3afcf3a8-5919-484c-b39f-4b1bd2784cef
0008: ce13c9e1-6cbd-42a4-9c15-294a99f3bde1
0009: 058aa52a-92f3-4686-bf1b-b4ab7a19167f
0010: 6fc7f0ae-d7bd-4feb-a776-8d38af5b5ed0
0011: 6834b460-aeec-402a-b770-c4fe5007954b
0012: c2105fce-cefd-48f8-9b30-c8fffc74f0b0
0013: 28138c82-e7f8-4dd0-a870-604f2b1292aa
0014: be9b0468-f66b-4477-8c78-abf28c1829ba
0015: be788208-358d-4591-a08a-0ebbe253b551
0016: e9537b87-b541-45a8-aa40-a4c11c31020f
0017: 231767dd-53a4-429c-9773-e474a698da65
0018: 9426efb5-ec68-4aab-99ae-4beb938c1bdc
0019: 9363b2a5-b95c-4bc5-a958-18edcc143e64
0020: dee22629-4858-4da4-834f-af4a8425635f
0021: e8ea7add-5083-4ce0-ba39-09898d94c72f
0022: 5f238c1e-8290-43a7-9d13-ba34798729d9
0023: 87fa6ec8-b63d-48cb-9210-2410

In [22]:
CHUNK_ID_KEY_IN_JSON = "chunk_ids"

CHUNK_IDS_JSON.write_text(
    json.dumps({CHUNK_ID_KEY_IN_JSON: chunk_ids}, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

pd.DataFrame({"chunk_id": chunk_ids}).to_csv(CHUNK_IDS_CSV, index=False)

print(f"Saved JSON: {CHUNK_IDS_JSON}")
print(f"Saved CSV: {CHUNK_IDS_CSV}")

Saved JSON: /home/aman/storage/Python/Projects/Research Paper Assistant/playground/chunk_ids.json
Saved CSV: /home/aman/storage/Python/Projects/Research Paper Assistant/playground/chunk_ids.csv


In [23]:
# Fetch exact content by point IDs from Qdrant (authoritative read).
point_ids = [r["point_id"] for r in chunk_records]

exact_points = []
BATCH = 128
for i in range(0, len(point_ids), BATCH):
    batch = point_ids[i:i + BATCH]
    # Qdrant accepts ids in native format; normalize numeric-looking ids.
    normalized_batch = [int(x) if str(x).isdigit() else x for x in batch]
    exact_points.extend(
        client.retrieve(
            collection_name=COLLECTION_NAME,
            ids=normalized_batch,
            with_payload=True,
            with_vectors=False,
        )
    )

exact_chunk_records = []
for p in exact_points:
    payload = p.payload if isinstance(p.payload, dict) else {}
    chunk_id = payload.get("chunk_id") or payload.get("_id")
    exact_chunk_records.append(
        {
            "point_id": str(p.id),
            "chunk_id": str(chunk_id) if chunk_id is not None else "",
            "section_title": str(payload.get("section_title") or ""),
            "section_id": str(payload.get("section_id") or ""),
            "content": str(payload.get("content") or ""),
            "payload": payload,
        }
    )

exact_df = pd.DataFrame(exact_chunk_records)
print(f"Exact chunks fetched by point_id: {len(exact_chunk_records)}")
exact_df[["point_id", "chunk_id", "section_title", "section_id", "content"]].head(20)

Exact chunks fetched by point_id: 96


,point_id,chunk_id,section_title,section_id,content
0,006558f7-0158-59e1-8a44-dea726b2cf43,12d0eec0-d010-4578-8595-95cc823bbe22,Reference,bd077a96-5a38-5281-993e-10cf869afcde_section_7,Diederik Kingma and Jimmy Ba. Adam: A method f...
1,0233c58f-c04c-5324-bf84-72532fdc1f24,a68ddde7-ac19-49ea-bae4-c82e5496244b,Reference,bd077a96-5a38-5281-993e-10cf869afcde_section_7,Smith. Recurrent neural network grammars. In P...
2,0278a119-a01b-539a-903f-65fa986c1bc1,2c4c7d66-0a9c-4c9d-b34e-5cb04da5df7a,Reference,bd077a96-5a38-5281-993e-10cf869afcde_section_7,Outrageously large neural networks: The sparse...
3,033d9657-784e-521f-ae38-8ca7d78fac12,1f1078a1-7b2d-408c-bec1-fe9d039eb9f1,Reference,bd077a96-5a38-5281-993e-10cf869afcde_section_7,"ACL, August 2013. Jimmy Lei Ba, Jamie Ryan Kir..."
4,0498359c-e94e-570a-8f23-692256a02687,c1af6cc1-824c-4122-9063-8c693ed63587,Reference,bd077a96-5a38-5281-993e-10cf869afcde_section_7,"ACL, June 2006. Ankur Parikh, Oscar Täckström,..."
5,07155d70-2e20-54aa-b25e-e6b10f079817,c2e45765-f41e-447f-95a6-0bd19be9244e,Reference,bd077a96-5a38-5281-993e-10cf869afcde_section_7,In Proceedings of the 2009 Conference on Empir...
6,0a213754-813f-5499-b38c-1c27e77755b6,3afcf3a8-5919-484c-b39f-4b1bd2784cef,Reference,bd077a96-5a38-5281-993e-10cf869afcde_section_7,Smith. Recurrent neural network grammars. In P...
7,15fa383b-bc84-5247-a86b-73d1380b2e80,ce13c9e1-6cbd-42a4-9c15-294a99f3bde1,Results,bd077a96-5a38-5281-993e-10cf869afcde_section_5,Table 3: Variations on the Transformer archite...
8,1937c5f0-a3a0-5230-9fa9-7fbd5af1f033,058aa52a-92f3-4686-bf1b-b4ab7a19167f,Why Self-Attention,bd077a96-5a38-5281-993e-10cf869afcde_section_3,Hence we also compare the maximum path length ...
9,1a540631-bff4-5989-94df-aae4f6e5a85a,6fc7f0ae-d7bd-4feb-a776-8d38af5b5ed0,Conclusion,bd077a96-5a38-5281-993e-10cf869afcde_section_6,We are excited about the future of attention-b...


In [24]:
section_titles = sorted({str(r.get("section_title") or "").strip() for r in exact_chunk_records})
non_empty_section_titles = [s for s in section_titles if s]
missing_section_title_count = sum(1 for r in exact_chunk_records if not str(r.get("section_title") or "").strip())

print(f"Unique non-empty section_title values: {len(non_empty_section_titles)}")
for title in non_empty_section_titles:
    print(f"- {title}")

print(f"Chunks with missing/empty section_title: {missing_section_title_count}")

Unique non-empty section_title values: 7
- Background
- Conclusion
- Introduction
- Model Architecture
- Reference
- Results
- Why Self-Attention
Chunks with missing/empty section_title: 0


In [25]:
grouped = defaultdict(list)
for rec in exact_chunk_records:
    key = str(rec.get("section_title") or "<MISSING_SECTION_TITLE>").strip() or "<MISSING_SECTION_TITLE>"
    grouped[key].append(rec)

section_export = {}
for section in sorted(grouped.keys()):
    rows = sorted(grouped[section], key=lambda x: (x.get("chunk_id", ""), x.get("point_id", "")))
    section_export[section] = [
        {
            "point_id": r["point_id"],
            "chunk_id": r["chunk_id"],
            "section_id": r["section_id"],
            "content": r["content"],
        }
        for r in rows
    ]

print(f"Sections found: {len(section_export)}")
for section, rows in section_export.items():
    print("\n" + "=" * 80)
    print(f"Section: {section} | chunks: {len(rows)}")
    print("=" * 80)
    for i, row in enumerate(rows, start=1):
        preview = " ".join(str(row["content"]).split())[:220]
        if len(str(row["content"])) > 220:
            preview += "..."
        print(f"{i:03d}. chunk_id={row['chunk_id']} | point_id={row['point_id']} | section_id={row['section_id']}")
        print(f"     content: {preview}")

SECTIONS_JSON.write_text(json.dumps(section_export, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"\nSaved grouped output: {SECTIONS_JSON}")

Sections found: 7

Section: Background | chunks: 6
001. chunk_id=08ac5732-d366-4fe7-8d79-a4fe3722d460 | point_id=8753155a-22db-5b6f-a825-959fa533081c | section_id=bd077a96-5a38-5281-993e-10cf869afcde_section_1
     content: The goal of reducing sequential computation also forms the foundation of the Extended Neural GPU [16], ByteNet [18] and ConvS2S [9], all of which use convolutional neural networks as basic building block, computing hidde...
002. chunk_id=23047bf8-0540-4887-b828-af805e65a0ce | point_id=a00a1b27-b1d5-5a6c-ab7b-9186fc86ba6f | section_id=bd077a96-5a38-5281-993e-10cf869afcde_section_1
     content: The goal of reducing sequential computation also forms the foundation of the Extended Neural GPU [16], ByteNet [18] and ConvS2S [9], all of which use convolutional neural networks as basic building block, computing hidde...
003. chunk_id=231767dd-53a4-429c-9773-e474a698da65 | point_id=273e1ffa-4374-5436-acab-3be7ad55351a | section_id=bd077a96-5a38-5281-993e-10cf869afcde_sectio